# Building AI Agents

### What You'll Learn

This notebook builds on your RAG knowledge and introduces **AI Agents** - intelligent systems that can reason, choose tools, and take actions autonomously. We'll take the same progressive, step-by-step approach:

1. **Basic Agent Concepts** - Understand what makes an agent "intelligent"
2. **From RAG to Agents** - Evolution of AI capabilities
3. **Tool Creation** - Define tools agents can use
4. **Simple Agent Building** - Create your first reasoning agent
5. **Agent Execution** - Watch agents think and act
6. **Adding Multiple Tools** - Expand agent capabilities
7. **Agent System** - Put it all together with LangGraph

### What is an AI Agent?

**An AI Agent** is an autonomous system that can:
- 🧠 **Reason** about problems and plan solutions
- 🔧 **Use tools** to gather information or take actions  
- 🔄 **Iterate** through multiple steps until goals are achieved
- 🎯 **Make decisions** about which tools to use when

Think of upgrading from a **smart search engine** (RAG) to a **digital assistant** that can actually do things!

### From RAG to Agents: The Evolution

| RAG System | Agent System |
|------------|--------------|
| Retrieves documents → Generates answers | Reasons → Chooses tools → Acts → Repeats |
| Single capability (document search) | Multiple capabilities (search, calculate, browse, etc.) |
| Predictable: always retrieves | Dynamic: decides what to do |
| Good for: Q&A from documents | Good for: Complex multi-step tasks |

### The Agent Architecture

**Key Components:**
- 🧠 **Reasoning Engine** - LLM that thinks through problems
- 🛠️ **Tool Registry** - Available actions (search, calculate, etc.)
- 🔄 **Execution Loop** - ReAct pattern (Reason → Act → Observe)
- 📝 **Memory** - Track conversation and previous actions

---

Let's build your first intelligent agent! 🤖

## Step 0: Environment Setup

Before we build agents, let's install all required packages. We'll need:

- **langgraph** - Framework for building stateful agent applications
- **langchain** - Core LLM framework and agent tools
- **langchain-openai** - Azure OpenAI integration (reusing from RAG)
- **tavily-python** - Real-time web search tool for agents
- **langchain-community** - Additional tools and integrations

Run the cell below to install all dependencies:

In [ ]:
%pip install langgraph langchain langchain-openai
%pip install langchain-community tavily-python
%pip install beautifulsoup4 faiss-cpu
# Note: gradio will be added later for the agent interface

## Step 1: Basic LLM Review - From RAG to Agents

Let's start by connecting to our LLM (same as RAG tutorial) and understand the key difference between simple generation and agent behavior.

### Simple LLM vs Agent LLM

- **Simple LLM**: Takes input → Generates text response
- **Agent LLM**: Takes input → Reasons → Chooses actions → Uses tools → Generates informed response

Let's see this in action:

In [ ]:
import os
import dotenv
from langchain_openai import AzureChatOpenAI

dotenv.load_dotenv()

# Reuse the same Azure OpenAI setup from RAG tutorial
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

print("✓ Connected to Azure OpenAI")

### Test: Simple LLM Question

Let's ask a question that requires current information - something a basic LLM can't answer accurately:

In [ ]:
# Ask about current information the LLM won't know
response = llm.invoke("What's the current weather in San Francisco today?")
print(f"Simple LLM Response:\n{response.content}")

In [ ]:
# Ask about a calculation
response = llm.invoke("What's 47 * 139 + 2847?")
print(f"Simple LLM Response:\n{response.content}")

### Understanding the Limitations

Notice the LLM's limitations:
- 🚫 **No real-time data** - Can't check current weather
- 🚫 **May make calculation errors** - Not always precise with math
- 🚫 **No external capabilities** - Can't browse web, access APIs, etc.

**This is where agents shine!** Agents can use tools to:
- ✅ Get real-time information (web search, APIs)
- ✅ Perform precise calculations
- ✅ Access external systems and databases
- ✅ Take actions in the real world

## Part 2: Creating Your First Tool 🔧

**Tools are the superpowers of agents!** They extend what an LLM can do by providing:
- Real-time data access
- Computational capabilities
- External system integration
- Action execution

Let's start by creating a simple calculator tool.

### 2.1 Install Required Packages

First, let's install the necessary packages:

### Create Your First Tool: Calculator

Tools in LangChain are just Python functions with special decorators that tell the LLM:
- **What the tool does** (description)
- **When to use it** (based on the description)
- **What inputs it needs** (parameters)

In [ ]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Performs mathematical calculations. 
    
    Use this tool when you need to do precise arithmetic.
    Input should be a mathematical expression like '47 * 139 + 2847'.
    """
    try:
        # Safely evaluate mathematical expressions
        result = eval(expression)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating {expression}: {str(e)}"

# Test the tool directly
print(calculator.invoke("47 * 139 + 2847"))


**Key Observations:**

1. **@tool decorator** - Tells LangChain this is an agent tool
2. **Clear docstring** - The LLM reads this to understand when to use the tool
3. **Type hints** - Help the LLM understand input/output format
4. **Error handling** - Tools should handle failures gracefully

Now let's create a search tool for real-time information!

### Create a Web Search Tool

For real-time information, we'll use **Tavily** - a search engine designed for AI agents.

**Why Tavily?**
- ✅ Optimized for LLMs (clean, summarized results)
- ✅ Fast and reliable
- ✅ Easy integration with LangChain

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Set up Tavily API key (get free key at https://tavily.com)
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY") or "your-tavily-api-key"

# Create the search tool
search_tool = TavilySearchResults(
    max_results=3,
    search_depth="basic",
    name="web_search",
    description="Search the web for current information, news, weather, and real-time data."
)

# Test the search tool
search_result = search_tool.invoke("current weather San Francisco")
print("Search Results:")
print(search_result)


## Step 3: Building Your First Agent

Now for the exciting part! We'll create an agent that can **reason** about which tool to use and **act** accordingly.

### The ReAct Pattern

Agents use the **ReAct** pattern:
1. **Reason** - Analyze the problem
2. **Act** - Choose and use a tool  
3. **Observe** - Review the results
4. **Repeat** - Continue until done

**Think of it like a smart assistant:**
- User: "What's 100 * 50 and what's the weather in Paris?"
- Agent: "I need to do math AND get current weather"
- Agent: Uses calculator for math, search for weather
- Agent: Combines results into a helpful response

### Create the Agent

LangGraph provides a simple way to create ReAct agents:

In [ ]:
from langgraph.prebuilt import create_react_agent

# Combine our tools
tools = [calculator, search_tool]

# Create the agent
agent = create_react_agent(llm, tools)

print("✓ Agent created with tools:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")


### Test the Agent with Simple Math

Let's start with something our calculator tool can handle:

In [ ]:
# Test with math calculation
response = agent.invoke({
    "messages": [("human", "What's 47 * 139 + 2847?")]
})

print("Agent Response:")
print(response["messages"][-1].content)


**Observe the magic!** 🪄

The agent:
1. **Reasoned** - "This is a math problem"
2. **Acted** - Called the calculator tool
3. **Observed** - Got the precise result
4. **Responded** - Provided the answer to the user

Much better than the LLM guessing at math!

### Test with Real-Time Information

Now let's try something that requires web search:

In [ ]:
# Test with current information
response = agent.invoke({
    "messages": [("human", "What's the current weather in San Francisco?")]
})

print("Agent Response:")
print(response["messages"][-1].content)


## Step 4: Multi-Tool Agent Execution

The real power of agents emerges when they need to use **multiple tools** for a single question. Let's test this:

In [ ]:
# Complex query requiring both tools
response = agent.invoke({
    "messages": [("human", "Calculate 25 * 144, and also tell me the current weather in Tokyo")]
})

print("Agent Response:")
print(response["messages"][-1].content)


**Incredible!** The agent:

1. **Parsed** the complex request into two tasks
2. **Used calculator** for the math (25 * 144)  
3. **Used web search** for Tokyo weather
4. **Combined results** into a coherent response

This is **multi-tool orchestration** - the agent figured out it needed both tools and used them appropriately!

---

## Congratulations! 🎉

You've successfully built your first AI Agent!

### What You've Accomplished

Let's recap this incredible journey:

1. **Basic Agent Concepts** ✓
   - Understood reasoning vs generation
   - Learned the ReAct pattern (Reason → Act → Observe)
   - Compared RAG systems to Agent systems

2. **Tool Creation** ✓
   - Built custom tools (calculator)
   - Integrated external tools (web search)
   - Learned tool principles

3. **Agent Building** ✓
   - Created your first ReAct agent
   - Watched agents choose appropriate tools
   - Understood autonomous decision making

4. **Multi-Tool Orchestration** ✓
   - Built agents that use multiple tools
   - Handled complex, multi-part queries
   - Observed intelligent tool selection

### Key Concepts you were exposed too

**Agent Benefits:**
- ✅ **Autonomous reasoning** - Thinks through problems step-by-step
- ✅ **Tool orchestration** - Uses multiple capabilities intelligently  
- ✅ **Real-time information** - Accesses current data via web search
- ✅ **Precise calculations** - No more LLM math errors
- ✅ **Extensible architecture** - Easy to add new capabilities

**Agent Architecture:**
```
User Query → Agent Reasoning → Tool Selection → Tool Execution → Result Synthesis → Response
```

### Next Steps: Advanced Agent Patterns

Ready to go further?

**📚 Next Notebook: `5.RAGWithAgents.ipynb`**

In the advanced notebook, you'll learn:
- ✅ **Conversational Memory** - Agents that remember previous interactions
- ✅ **Streaming & Observability** - Watch agents think in real-time
- ✅ **Agent Supervision** - Multi-agent orchestration patterns
- ✅ **Production Deployment** - Deploy agents with LangServe
- ✅ **Advanced Tool Patterns** - File operations, database queries, API integrations

### The Agent Advantage

**Traditional LLM:**
- Single-step: Input → Output
- Limited to training knowledge
- No external capabilities

**AI Agent:**
- Multi-step: Reason → Act → Observe → Repeat
- Access to real-time information
- Extensible tool ecosystem
- Dynamic problem-solving

**Happy Agent Building!** 🤖✨